In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080304

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] exp01_env
[学号]       20244080304
[机器指纹] 5fc15abbc5865ff7
[RUN_ID]   20244080304-5fc15abbc5865ff7
[Python]    3.10.22


# 实验报告03：正则表达式与网络结构化数据

学号：20244080304　姓名：李晓月

## 实验1：正则表达式的基础匹配

In [2]:
import re

text = "cat on mat; later: cats and dogs"
print(r"\d+：", re.findall(r"\d+", text))
print(r"\s+：", re.findall(r"\s+", text))
print(r"\bcat\b：", re.findall(r"\bcat\b", text))

\d+： []
\s+： [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b： ['cat']


In [3]:
text = "cat dog bird CAT"
print(r"cat|dog：", re.findall(r"cat|dog", text))
print(r"a|aa|aaa：", re.findall(r"a|aa|aaa", "aaaa"))
print(r"aaa|aa|a：", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog： ['cat', 'dog']
a|aa|aaa： ['a', 'a', 'a', 'a']
aaa|aa|a： ['aaa', 'a']


In [4]:
text = "ct cat caat caaat"
for pattern in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(f"{pattern:9s}", re.findall(pattern, text))

ca*t      ['ct', 'cat', 'caat', 'caaat']
ca+t      ['cat', 'caat', 'caaat']
ca?t      ['ct', 'cat']
ca{2}t    ['caat']
ca{1,2}t  ['cat', 'caat']


`*` 表示前一个字符出现 0 次或多次，`+` 表示 1 次或多次，`?` 表示 0 次或 1 次，`{2}` 表示恰好 2 次，`{1,2}` 表示 1 到 2 次。因此 `ca*t` 还能匹配 `ct`，`ca?t` 不能匹配两个及以上 `a`，`ca{2}t` 只匹配 `caat`。

`a|aa|aaa` 在 `aaaa` 上每次先取一个 `a`，得到四个 `a`；`aaa|aa|a` 先取三个 `a`，剩下一个 `a`。

## 实验2：分组、回溯引用与替换

In [5]:
text = "联系：alice@example.com；备选：bob@test.org"
pattern = re.compile(r"([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)")
for match in pattern.finditer(text):
    print("用户名：", match.group(1), "域名：", match.group(2))
print(pattern.findall(text))

用户名： alice 域名： example.com
用户名： bob 域名： test.org
[('alice', 'example.com'), ('bob', 'test.org')]


In [6]:
text = "cat cat, dog cat, bird bird"
pattern = re.compile(r"\b(\w+)\s+\1\b")
print("重复的单词：", pattern.findall(text))
print("替换后：", pattern.sub(r"\1", text))

重复的单词： ['cat', 'bird']
替换后： cat, dog cat, bird


In [7]:
text = "2024-05-01\n2024/05/02\n2024.05.03"
pattern = re.compile(r"(\d{4})([-/.])(\d{2})\2(\d{2})")
print("转换前：")
print(text)
print("转换后：")
print(pattern.sub(r"\3-\4-\1", text))

转换前：
2024-05-01
2024/05/02
2024.05.03
转换后：
05-01-2024
05-02-2024
05-03-2024


In [8]:
text = "联系我：123-456-7890 或 987.654.3210"
pattern = re.compile(r"(\d{3})[-.](\d{3})[-.](\d{4})")
print(pattern.sub(r"(\1) \2-\3", text))

联系我：(123) 456-7890 或 (987) 654-3210


`\1`、`\2` 等是回溯引用，分别表示前面第 1、2 个捕获组的内容。日期替换用 `\2` 保证年、月、日之间使用同一种分隔符；电话号码用三个分组重新拼成 `(123) 456-7890`。

## 实验3：使用 BeautifulSoup 解析 HTML

In [9]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")
print("标题：", soup.title.get_text())
print("h1 标签名：", soup.h1.name)
print("h1 文本：", soup.h1.get_text())

标题： 实验课程
h1 标签名： h1
h1 文本： 数据分析


In [10]:
for p in soup.find_all("p", class_="course"):
    print(p.get_text())
for a in soup.find_all("a"):
    print(a.get_text(), a["href"])

第六讲：正则表达式


第七讲：结构化数据
讲义六 https://example.org/lecture06
讲义七 https://example.org/lecture07


In [11]:
whole = soup.get_text()
para = soup.find("p", class_="course").get_text()
link = soup.find("a").get_text()
print("整个文档：", repr(whole))
print("单个段落：", repr(para))
print("单个链接：", repr(link))
print("长度：", len(whole), len(para), len(link))

整个文档： '\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'
单个段落： '第六讲：正则表达式'
单个链接： '讲义六'
长度： 40 9 3


`get_text()` 只返回标签内的可见文本，不包含标签本身；对整篇文档调用会拼接所有文本并保留元素间的空白，对单个元素调用只返回该元素的文字。